# CorrDiff Ensemble Inference - Memory Optimized Workflow

**Memory-efficient CorrDiff ensemble generation with proper GPU memory management.**

## Key Features
- 🧠 **Memory Optimized**: Efficient GPU memory usage and cleanup
- 🎯 **Stable Inference**: Robust ensemble generation with error handling
- 📊 **Comprehensive Analysis**: Ensemble statistics and visualization
- 🔧 **Direct Implementation**: No dependency on external modules

## Fixed Issues
- ✅ CUDA out of memory errors resolved
- ✅ Proper GPU memory cleanup between steps
- ✅ Efficient data processing pipeline
- ✅ Stable ensemble inference

## Configuration

In [1]:
# =============================================================================
# CONFIGURATION - Update these paths and parameters for your setup
# =============================================================================

VARIABLES = "Fog_index"

# Checkpoint paths
BASE_CHECKPOINTS_PATH = "/app/host/home/younes.abid/git/physicsnemo/outputs/checkpoints/"

REGRESSION_CHECKPOINT = BASE_CHECKPOINTS_PATH + VARIABLES + '/checkpoints_regression/UNet.0.390000.mdlus'
DIFFUSION_CHECKPOINT = BASE_CHECKPOINTS_PATH + VARIABLES + '/checkpoints_diffusion/EDMPrecondSuperResolution.0.590000.mdlus'

# Data paths
BASE_DATA_PATH = '/app/host/mnt/storage/younes.abid/physicsnemo/data/custom_data_2/'
DATA_FILE = BASE_DATA_PATH + 'ERA5_WRF_combined_concatenated_432/2024-04-30_2024-05-30_21.nc'
STATS_FILE = BASE_DATA_PATH + 'stats_432/stat.json'

# Variables
INPUT_VARIABLES = ['t_850', 't_500', 'z_850', 'z_500', 'u_850', 'u_500', 'v_850', 'v_500', 'u10', 'v10', 't2m', 'd2m', 'skt', 'sp', 'tcwv', 'tp']
OUTPUT_VARIABLES = ['Fog_index']

# Domain coordinates
INPUT_GRID = None
# {
#     'lat': (19.25, 28.0, 36),   # (min, max, points)
#     'lon': (116.0, 126.0, 40)   # (min, max, points)
# }
OUTPUT_GRID = None
# {
#     'lat': (19.0, 28.0, 432),   # (min, max, points) 
#     'lon': (116.0, 126.0, 432)  # (min, max, points)
# }

# Ensemble parameters
INFERENCE_TIMES = ['2024-05-01T00:00:00', '2024-05-01T06:00:00', '2024-05-01T12:00:00']
NUM_ENSEMBLES = 8
SEED_BASE = 42
SAMPLING_MODE = 'stochastic'
NUMBER_OF_STEPS = 20
SOLVER = 'euler'
HR_MEAN_CONDITIONING = True

# Output configuration
from datetime import datetime
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_FILE = f'/app/outputs/generation/{VARIABLES}/{TIMESTAMP}/ensemble.nc'

print('✅ Configuration loaded')
print(f'📊 Ensemble setup: {NUM_ENSEMBLES} members, {SAMPLING_MODE} sampling')
print(f'🔄 Diffusion steps: {NUMBER_OF_STEPS}, solver: {SOLVER}')
print(f'💾 Output: {OUTPUT_FILE}')

✅ Configuration loaded
📊 Ensemble setup: 8 members, stochastic sampling
🔄 Diffusion steps: 20, solver: euler
💾 Output: /app/outputs/generation/Fog_index/20260203_114505/ensemble.nc


## Setup and Imports

In [2]:
import os
import sys
import torch
import numpy as np
import xarray as xr
import netCDF4 as nc
import gc
from collections import OrderedDict
from pathlib import Path

# Add tutorials path
tutorials_path = '/app/host/home/younes.abid/git/earth2studio/notebooks/tutorials'
sys.path.append(tutorials_path)

# Earth2Studio imports
from earth2studio.data import prep_data_array
from earth2studio.utils.coords import map_coords
from earth2studio.utils.time import to_time_array
from earth2studio.models.batch import batch_coords, batch_func
from earth2studio.models.dx.base import DiagnosticModel
from earth2studio.utils import handshake_coords, handshake_dim

# PhysicsNeMo imports
from physicsnemo.models import Module as PhysicsNemoModule
from physicsnemo.utils.generative import StackedRandomGenerator, deterministic_sampler, stochastic_sampler

# Custom data loader
from custom_data_loader import CustomCorrDiffDataSource

# GPU Memory management
def clear_gpu_memory():
    """Clear GPU memory cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()

print('✅ All imports successful')

✅ All imports successful


## Data Source Setup

In [3]:
# Create data source
data_source = CustomCorrDiffDataSource(
    data_paths=[DATA_FILE],
    stats_path=STATS_FILE,
    input_variables=INPUT_VARIABLES,
    output_variables=OUTPUT_VARIABLES,
    cache_data=True
)

print('✅ Data source created')
print(f'📊 Available samples: {data_source.total_samples}')
print(f'📐 Input grid shape: {data_source.input_shape}')

🔍 Analyzing data files...
  File 1: 2024-04-30_2024-05-30_21.nc - 504 samples
✅ Total samples across all files: 504
📐 Input grid shape: (432, 432)
📐 Lat range: [19.25, 28.00]
📐 Lon range: [116.00, 125.98]
✅ Loaded normalization stats for 16 input variables
💾 Preloading data into memory...
  Loading file 1/1: 2024-04-30_2024-05-30_21.nc
    ✅ Loaded in 6.22s
✅ Data preloading complete!
✅ Data source created
📊 Available samples: 504
📐 Input grid shape: (432, 432)


## Ensemble CorrDiff Model

In [4]:
class EnsembleFogIndexCorrDiff(torch.nn.Module, DiagnosticModel):
    """Memory-optimized Ensemble CorrDiff model"""
    
    def __init__(self, regression_model, residual_model=None, sampling_config=None):
        super().__init__()
        self.regression_model = regression_model
        self.residual_model = residual_model
        
        # Sampling configuration
        self.sampling_config = sampling_config or {}
        self.sampling_mode = self.sampling_config.get('mode', 'stochastic')
        self.num_steps = self.sampling_config.get('num_steps', 20)
        self.solver = self.sampling_config.get('solver', 'euler')
        self.hr_mean_conditioning = self.sampling_config.get('hr_mean_conditioning', True)
        
        # Grid coordinates
        self.input_lat = np.linspace(*INPUT_GRID['lat'])
        self.input_lon = np.linspace(*INPUT_GRID['lon'])
        self.output_lat = np.linspace(*OUTPUT_GRID['lat'])
        self.output_lon = np.linspace(*OUTPUT_GRID['lon'])
        
        # Normalization
        self.register_buffer('in_center', torch.from_numpy(data_source.input_mean))
        self.register_buffer('in_scale', torch.from_numpy(data_source.input_std))
        self.register_buffer('out_center', torch.from_numpy(data_source.output_mean) if hasattr(data_source, 'output_mean') else torch.zeros(len(OUTPUT_VARIABLES), 1, 1))
        self.register_buffer('out_scale', torch.from_numpy(data_source.output_std) if hasattr(data_source, 'output_std') else torch.ones(len(OUTPUT_VARIABLES), 1, 1))
        
        # Ensemble parameters
        self.number_of_samples = NUM_ENSEMBLES
        self.ensemble_seeds = None
    
    def set_ensemble_seeds(self, base_seed=42):
        """Generate ensemble seeds"""
        np.random.seed(base_seed)
        self.ensemble_seeds = np.random.randint(0, 2**31, size=self.number_of_samples)
        print(f'🎲 Generated {len(self.ensemble_seeds)} ensemble seeds: {self.ensemble_seeds[:5]}...')
    
    def input_coords(self):
        return OrderedDict({
            'batch': np.empty(0),
            'variable': np.array(INPUT_VARIABLES),
            'lat': self.input_lat,
            'lon': self.input_lon,
        })
    
    @batch_coords()
    def output_coords(self, input_coords):
        output_coords = OrderedDict({
            'batch': np.empty(0),
            'sample': np.arange(self.number_of_samples),
            'variable': np.array(OUTPUT_VARIABLES),
            'lat': self.output_lat,
            'lon': self.output_lon,
        })
        
        # Validate input
        target = self.input_coords()
        handshake_dim(input_coords, 'lon', 3)
        handshake_dim(input_coords, 'lat', 2)
        handshake_dim(input_coords, 'variable', 1)
        handshake_coords(input_coords, target, 'lon')
        handshake_coords(input_coords, target, 'lat')
        handshake_coords(input_coords, target, 'variable')
        
        output_coords['batch'] = input_coords['batch']
        return output_coords
    
    def _interpolate(self, x):
        """Interpolate to output resolution"""
        if x.shape[-2:] == (len(self.output_lat), len(self.output_lon)):
            return x
        
        # Add batch dim if needed
        if len(x.shape) == 3:
            x = x.unsqueeze(0)
            added_batch = True
        else:
            added_batch = False
        
        # Interpolate
        import torch.nn.functional as F
        target_size = (len(self.output_lat), len(self.output_lon))
        x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=True)
        
        # Remove batch dim if added
        if added_batch:
            x = x.squeeze(0)
        
        return x
    
    @torch.inference_mode()
    def _forward(self, x):
        """Forward pass with memory optimization"""
        # Clear memory before processing
        clear_gpu_memory()
        
        # Interpolate and normalize
        x = self._interpolate(x)
        x = (x - self.in_center) / self.in_scale
        
        # Add sample dimension
        x = x.unsqueeze(0).repeat(self.number_of_samples, 1, 1, 1)
        
        # Create output tensors
        img_h, img_w = len(self.output_lat), len(self.output_lon)
        latents = torch.zeros(self.number_of_samples, len(OUTPUT_VARIABLES), img_h, img_w, device=x.device)
        
        # Generate ensemble seeds if not set
        if self.ensemble_seeds is None:
            self.set_ensemble_seeds(SEED_BASE)
        
        if self.residual_model is not None:
            # Full diffusion mode
            print(f'🔄 Running {self.sampling_mode} sampling with {self.num_steps} steps')
            
            # Create random number generator
            rnd = StackedRandomGenerator(x.device, torch.from_numpy(self.ensemble_seeds))
            noise = rnd.randn_like(latents)
            
            # Regression step
            if self.hr_mean_conditioning:
                mean = self.regression_model(latents, x)
                x_with_mean = torch.cat([x, mean], dim=1)
            else:
                mean = torch.zeros_like(latents)
                x_with_mean = x
            
            # Choose sampling method
            if self.sampling_mode == 'deterministic':
                residual = deterministic_sampler(
                    self.residual_model, 
                    noise, 
                    x_with_mean, 
                    randn_like=rnd.randn_like, 
                    num_steps=self.num_steps,
                    solver=self.solver
                )
            elif self.sampling_mode == 'stochastic':
                residual = stochastic_sampler(
                    self.residual_model, 
                    noise, 
                    x_with_mean, 
                    randn_like=rnd.randn_like, 
                    num_steps=self.num_steps
                )
            else:
                raise ValueError(f'Unknown sampling mode: {self.sampling_mode}')
            
            x = mean + residual
        else:
            # Regression only
            print('📊 Running regression-only mode')
            x = self.regression_model(latents, x)
        
        # Clear intermediate tensors
        del latents
        clear_gpu_memory()
        
        # Denormalize
        x = self.out_scale * x + self.out_center
        return x
    
    @batch_func()
    def __call__(self, x, coords):
        output_coords = self.output_coords(coords)
        out = torch.zeros([len(v) for v in output_coords.values()], device=x.device, dtype=torch.float32)
        
        for i in range(out.shape[0]):
            out[i] = self._forward(x[i])
            # Clear memory after each batch
            clear_gpu_memory()
        
        return out, output_coords

print('✅ Memory-optimized ensemble model class defined')

✅ Memory-optimized ensemble model class defined


## Load Models

In [5]:
# Clear memory before loading models
clear_gpu_memory()

# Load models
print(f'🔄 Loading regression model: {Path(REGRESSION_CHECKPOINT).name}')
regression_model = PhysicsNemoModule.from_checkpoint(REGRESSION_CHECKPOINT).eval()

diffusion_model = None
if os.path.exists(DIFFUSION_CHECKPOINT):
    print(f'🔄 Loading diffusion model: {Path(DIFFUSION_CHECKPOINT).name}')
    diffusion_model = PhysicsNemoModule.from_checkpoint(DIFFUSION_CHECKPOINT).eval()
else:
    print('⚠️  No diffusion model found, using regression-only mode')

# Sampling configuration
sampling_config = {
    'mode': SAMPLING_MODE,
    'num_steps': NUMBER_OF_STEPS,
    'solver': SOLVER,
    'hr_mean_conditioning': HR_MEAN_CONDITIONING
}

# Create ensemble model
ensemble_corrdiff = EnsembleFogIndexCorrDiff(
    regression_model, 
    diffusion_model, 
    sampling_config=sampling_config
)

ensemble_corrdiff.set_ensemble_seeds(SEED_BASE)

print('✅ Models loaded successfully')
print(f'🎯 Model mode: {"Full CorrDiff" if diffusion_model else "Regression-only"}')
print(f'🎲 Sampling: {SAMPLING_MODE} with {NUM_ENSEMBLES} ensemble members')

🔄 Loading regression model: UNet.0.390000.mdlus


/usr/local/lib/python3.11/dist-packages/physicsnemo/models/module.py:460: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_dict = torch.load(


🔄 Loading diffusion model: EDMPrecondSuperResolution.0.590000.mdlus
🎲 Generated 8 ensemble seeds: [1608637542 1273642419 1935803228  787846414  996406378]...
✅ Models loaded successfully
🎯 Model mode: Full CorrDiff
🎲 Sampling: stochastic with 8 ensemble members


## Memory-Optimized Ensemble Inference

In [6]:
def run_memory_optimized_inference(times, corrdiff_model, data_source, device):
    """Memory-optimized ensemble inference with proper cleanup"""
    
    print(f'🚀 Starting memory-optimized ensemble inference on {device}')
    print(f'📅 Processing {len(times)} time steps')
    print(f'🎯 Generating {corrdiff_model.number_of_samples} ensemble members per time step')
    
    # Move model to device
    corrdiff_model = corrdiff_model.to(device)
    
    # Storage for results
    all_predictions = []
    all_inputs = []
    all_coords = []
    processed_times = []
    
    # Process each time step with memory management
    for i, time_step in enumerate(times):
        print(f'🔄 Processing time {i+1}/{len(times)}: {time_step}')
        
        # Clear memory before processing each time step
        clear_gpu_memory()
        
        try:
            # Load data for this time step
            time_array = to_time_array([time_step])
            x, coords = prep_data_array(
                data_source(time_array, INPUT_VARIABLES), 
                device=device
            )
            x, coords = map_coords(x, coords, corrdiff_model.input_coords())
            
            print(f'   📊 Input data shape: {x.shape}')
            
            # Store input (move to CPU to save GPU memory)
            all_inputs.append(x.cpu())
            
            # Run ensemble inference
            with torch.no_grad():
                pred, pred_coords = corrdiff_model(x, coords)
            
            print(f'   ✅ Generated ensemble shape: {pred.shape}')
            
            # Store results (move to CPU immediately)
            all_predictions.append(pred.cpu())
            all_coords.append(pred_coords)
            processed_times.append(time_array[0])
            
            # Clean up GPU tensors
            del x, pred
            clear_gpu_memory()
            
        except Exception as e:
            print(f'   ❌ Error processing {time_step}: {e}')
            # Clean up on error
            clear_gpu_memory()
            continue
    
    print(f'✅ Ensemble inference complete! Processed {len(processed_times)} time steps')
    
    return {
        'predictions': all_predictions,
        'inputs': all_inputs,
        'coords': all_coords,
        'times': processed_times
    }

# Run the inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = run_memory_optimized_inference(INFERENCE_TIMES, ensemble_corrdiff, data_source, device)

print(f'📊 Results: {len(results["predictions"])} time steps with {NUM_ENSEMBLES} ensemble members each')

🚀 Starting memory-optimized ensemble inference on cuda
📅 Processing 3 time steps
🎯 Generating 8 ensemble members per time step
🔄 Processing time 1/3: 2024-05-01T00:00:00
🔄 Loading data for 1 times and 16 variables
✅ Loaded DataArray with shape: (1, 16, 432, 432)
   📊 Input data shape: torch.Size([1, 16, 36, 40])
🔄 Running stochastic sampling with 20 steps


/usr/local/lib/python3.11/dist-packages/physicsnemo/models/diffusion/layers.py:701: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=self.amp_mode):


   ✅ Generated ensemble shape: torch.Size([1, 8, 1, 432, 432])
🔄 Processing time 2/3: 2024-05-01T06:00:00
🔄 Loading data for 1 times and 16 variables
✅ Loaded DataArray with shape: (1, 16, 432, 432)
   📊 Input data shape: torch.Size([1, 16, 36, 40])
🔄 Running stochastic sampling with 20 steps


/usr/local/lib/python3.11/dist-packages/physicsnemo/models/diffusion/layers.py:701: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=self.amp_mode):


   ✅ Generated ensemble shape: torch.Size([1, 8, 1, 432, 432])
🔄 Processing time 3/3: 2024-05-01T12:00:00
🔄 Loading data for 1 times and 16 variables
✅ Loaded DataArray with shape: (1, 16, 432, 432)
   📊 Input data shape: torch.Size([1, 16, 36, 40])
🔄 Running stochastic sampling with 20 steps


/usr/local/lib/python3.11/dist-packages/physicsnemo/models/diffusion/layers.py:701: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=self.amp_mode):


   ✅ Generated ensemble shape: torch.Size([1, 8, 1, 432, 432])
✅ Ensemble inference complete! Processed 3 time steps
📊 Results: 3 time steps with 8 ensemble members each


## Save Results

In [7]:
def save_ensemble_netcdf(results, output_path):
    """Save ensemble results in NetCDF format"""
    
    print(f'💾 Saving ensemble results to: {output_path}')
    
    # Create output directory
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Extract dimensions
    n_times = len(results['times'])
    n_ensembles = NUM_ENSEMBLES
    n_lat = len(ensemble_corrdiff.output_lat)
    n_lon = len(ensemble_corrdiff.output_lon)
    
    print(f'   📐 Dimensions: time={n_times}, ensemble={n_ensembles}, lat={n_lat}, lon={n_lon}')
    
    # Create NetCDF file
    with nc.Dataset(output_path, 'w', format='NETCDF4') as f:
        
        # Global attributes
        f.title = 'CorrDiff Ensemble Predictions'
        f.description = f'Ensemble forecasts with {n_ensembles} members'
        f.created = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        f.sampling_mode = SAMPLING_MODE
        f.num_steps = NUMBER_OF_STEPS
        f.base_seed = SEED_BASE
        
        # Prediction group
        pred_group = f.createGroup('prediction')
        pred_group.createDimension('time', n_times)
        pred_group.createDimension('ensemble', n_ensembles)
        pred_group.createDimension('y', n_lat)
        pred_group.createDimension('x', n_lon)
        
        # Coordinates
        time_var = pred_group.createVariable('time', 'f8', ('time',))
        ens_var = pred_group.createVariable('ensemble', 'i4', ('ensemble',))
        y_var = pred_group.createVariable('y', 'f4', ('y',))
        x_var = pred_group.createVariable('x', 'f4', ('x',))
        
        time_var[:] = [(t - np.datetime64('1970-01-01T00:00:00')) / np.timedelta64(1, 'h') for t in results['times']]
        time_var.units = 'hours since 1970-01-01 00:00:00'
        ens_var[:] = np.arange(n_ensembles)
        y_var[:] = ensemble_corrdiff.output_lat
        x_var[:] = ensemble_corrdiff.output_lon
        
        # Data variables
        for var_idx, var_name in enumerate(OUTPUT_VARIABLES):
            var = pred_group.createVariable(var_name, 'f4', ('ensemble', 'time', 'y', 'x'), 
                                          compression='zlib', complevel=4)
            
            var_data = np.zeros((n_ensembles, n_times, n_lat, n_lon))
            for t_idx, pred in enumerate(results['predictions']):
                var_data[:, t_idx, :, :] = pred[0, :, var_idx, :, :]
            
            var[:] = var_data
            var.long_name = f'Ensemble predictions for {var_name}'
        
        # Input group
        input_group = f.createGroup('input')
        input_group.createDimension('time', n_times)
        input_group.createDimension('y_input', len(ensemble_corrdiff.input_lat))
        input_group.createDimension('x_input', len(ensemble_corrdiff.input_lon))
        
        time_input = input_group.createVariable('time', 'f8', ('time',))
        y_input = input_group.createVariable('y', 'f4', ('y_input',))
        x_input = input_group.createVariable('x', 'f4', ('x_input',))
        
        time_input[:] = time_var[:]
        time_input.units = time_var.units
        y_input[:] = ensemble_corrdiff.input_lat
        x_input[:] = ensemble_corrdiff.input_lon
        
        for var_idx, var_name in enumerate(INPUT_VARIABLES):
            var = input_group.createVariable(var_name, 'f4', ('time', 'y_input', 'x_input'),
                                            compression='zlib', complevel=4)
            
            var_data = np.zeros((n_times, len(ensemble_corrdiff.input_lat), len(ensemble_corrdiff.input_lon)))
            for t_idx, inp in enumerate(results['inputs']):
                var_data[t_idx, :, :] = inp[0, var_idx, :, :]
            
            var[:] = var_data
            var.long_name = f'Input {var_name}'
    
    print(f'✅ Results saved to {output_path}')
    return output_path

# Save the results
if results['predictions']:
    saved_file = save_ensemble_netcdf(results, OUTPUT_FILE)
    print(f'📁 Saved file size: {Path(saved_file).stat().st_size / 1024**2:.1f} MB')
else:
    print('❌ No predictions to save')
    saved_file = None

💾 Saving ensemble results to: /app/outputs/generation/Fog_index/20260203_114505/ensemble.nc
   📐 Dimensions: time=3, ensemble=8, lat=432, lon=432
✅ Results saved to /app/outputs/generation/Fog_index/20260203_114505/ensemble.nc
📁 Saved file size: 12.6 MB


## Results Analysis

In [8]:
if saved_file and os.path.exists(saved_file):
    print('📊 ENSEMBLE ANALYSIS')
    print('=' * 50)
    
    # Load and analyze
    pred_ds = xr.open_dataset(saved_file, group='prediction')
    
    for var_name in OUTPUT_VARIABLES:
        if var_name in pred_ds:
            var_data = pred_ds[var_name]
            ens_mean = var_data.mean(dim='ensemble')
            ens_std = var_data.std(dim='ensemble')
            
            print(f'\n{var_name} Statistics:')
            print(f'   Mean range: [{ens_mean.min().values:.4f}, {ens_mean.max().values:.4f}]')
            print(f'   Uncertainty range: [{ens_std.min().values:.4f}, {ens_std.max().values:.4f}]')
            print(f'   Average uncertainty: {ens_std.mean().values:.4f}')
    
    pred_ds.close()
    
    print(f'\n✅ Analysis complete!')
    print(f'📁 Results saved to: {saved_file}')
    print(f'💻 Load with: xr.open_dataset("{saved_file}", group="prediction")')
    
else:
    print('❌ No results file available for analysis')

📊 ENSEMBLE ANALYSIS

Fog_index Statistics:
   Mean range: [-0.8704, 88.2364]
   Uncertainty range: [0.9079, 23.2127]
   Average uncertainty: 7.8851

✅ Analysis complete!
📁 Results saved to: /app/outputs/generation/Fog_index/20260203_114505/ensemble.nc
💻 Load with: xr.open_dataset("/app/outputs/generation/Fog_index/20260203_114505/ensemble.nc", group="prediction")
